# Chapter 1 Reusable Template
### Correlation, Association, and the Yule–Simpson Paradox

This notebook is a **reusable analysis template** for applying the ideas in Chapter 1 of *A First Course in Causal Inference* to your own project data.

**What it gives you:**
1. Functions to compute risk difference (rd), risk ratio (rr), and odds ratio (or) from a 2x2 table
2. A function to test association (Fisher's exact / chi-square) in a 2x2 table
3. A function to run a **stratified analysis** and automatically flag a Yule–Simpson Paradox
4. A regression comparison template (naive vs. covariate-adjusted)
5. A plotting function to visualize the geometry of Simpson's Paradox

Replace the example data in each section with your own `Z` (treatment/exposure), `Y` (outcome), and `X` (candidate confounder) variables.

**How to use this template for a new project:**
- Fill in Section 1 with your raw data (CSV or pandas DataFrame).
- Run Section 2 to get marginal (unadjusted) association measures.
- Run Section 3 to check whether a candidate confounder reverses or changes the association.
- Run Section 4 if you have continuous outcomes (regression-based).
- Use Section 5 to visualize and communicate the paradox/confounding to stakeholders.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option('display.precision', 4)
%matplotlib inline

## 1. Load your data

Your dataset should have at minimum:
- `Z`: a binary treatment/exposure indicator (1/0)
- `Y`: a binary outcome indicator (1/0) -- for the 2x2-table tools below
- `X`: one or more candidate confounders/stratifying variables

Replace the toy example below (the kidney stone data from the chapter) with `pd.read_csv('your_file.csv')`.

In [ ]:
# EXAMPLE: kidney stone data from Charig et al. (1986), used in Section 1.3 of the chapter
# Z = 1: open surgery, Z = 0: small puncture; Y = 1: success, Y = 0: failure
# X = 1: smaller stones, X = 0: larger stones
data = pd.DataFrame({
    'Z': [1]*273 + [1]*77 + [0]*289 + [0]*61,
    'Y': [1]*273 + [0]*77 + [1]*289 + [0]*61,
})
# Reconstruct stratum X consistent with the two sub-tables in the book
# (81,6) small-stone/open; (192,71) large-stone/open; (234,36) small-stone/puncture; (55,25) large-stone/puncture
rows = []
for _ in range(81):  rows.append((1, 1, 1))   # Z=1,Y=1,X=1 (small)
for _ in range(6):   rows.append((1, 0, 1))
for _ in range(192): rows.append((1, 1, 0))   # X=0 (large)
for _ in range(71):  rows.append((1, 0, 0))
for _ in range(234): rows.append((0, 1, 1))
for _ in range(36):  rows.append((0, 0, 1))
for _ in range(55):  rows.append((0, 1, 0))
for _ in range(25):  rows.append((0, 0, 0))
data = pd.DataFrame(rows, columns=['Z', 'Y', 'X'])
data.head()

## 2. Association measures for a 2x2 table (Section 1.2.2)

`rd`, `rr`, and `or` as defined in Proposition 1.1, plus a significance test.

In [ ]:
def two_by_two(df, z_col='Z', y_col='Y'):
    """Build the 2x2 count table: rows Z=1/Z=0, columns Y=1/Y=0."""
    n11 = ((df[z_col] == 1) & (df[y_col] == 1)).sum()
    n10 = ((df[z_col] == 1) & (df[y_col] == 0)).sum()
    n01 = ((df[z_col] == 0) & (df[y_col] == 1)).sum()
    n00 = ((df[z_col] == 0) & (df[y_col] == 0)).sum()
    return np.array([[n11, n10], [n01, n00]])

def association_measures(tb):
    """Given a 2x2 numpy array [[n11,n10],[n01,n00]], return rd, rr, or, and tests.
    Mirrors Section 1.2.2 of the chapter."""
    n11, n10 = tb[0]
    n01, n00 = tb[1]
    p1 = n11 / (n11 + n10)   # pr(Y=1 | Z=1)
    p0 = n01 / (n01 + n00)   # pr(Y=1 | Z=0)
    rd = p1 - p0
    rr = p1 / p0 if p0 != 0 else np.nan
    orr = (n11 * n00) / (n10 * n01) if (n10 * n01) != 0 else np.nan

    chi2, chi2_p, _, _ = stats.chi2_contingency(tb, correction=False)
    odds_ratio_exact, fisher_p = stats.fisher_exact(tb)

    return {
        'p(Y=1|Z=1)': p1, 'p(Y=1|Z=0)': p0,
        'risk_difference': rd, 'risk_ratio': rr, 'odds_ratio': orr,
        'chi2_stat': chi2, 'chi2_pvalue': chi2_p,
        'fisher_odds_ratio': odds_ratio_exact, 'fisher_pvalue': fisher_p
    }

marginal_table = two_by_two(data)
print('Marginal (aggregated) 2x2 table [[n11,n10],[n01,n00]]:')
print(marginal_table)
pd.Series(association_measures(marginal_table))

## 3. Stratified analysis & automatic Simpson's Paradox check (Section 1.3)

This is the core reusable tool: it computes the marginal association and the
association within each stratum of a candidate confounder `X`, then flags
whether the Yule–Simpson Paradox (sign reversal) is present.

In [ ]:
def stratified_analysis(df, z_col='Z', y_col='Y', x_col='X', measure='risk_difference'):
    """Compare the marginal association measure to the stratum-specific measures.
    Returns a DataFrame and prints a paradox flag."""
    results = {}
    marg_tb = two_by_two(df, z_col, y_col)
    results['Marginal (all data)'] = association_measures(marg_tb)[measure]

    for level, sub in df.groupby(x_col):
        tb = two_by_two(sub, z_col, y_col)
        results[f'{x_col} = {level}'] = association_measures(tb)[measure]

    out = pd.Series(results, name=measure).to_frame()

    marg_sign = np.sign(out.loc['Marginal (all data)', measure])
    strat_signs = np.sign(out.drop('Marginal (all data)')[measure])
    paradox = (strat_signs != 0).all() and (strat_signs == -marg_sign).all() and marg_sign != 0

    print(f"Yule–Simpson Paradox detected on '{measure}': {paradox}")
    if paradox:
        print(f"  -> '{x_col}' reverses the sign of the {z_col}-{y_col} association;")
        print(f"     treat '{x_col}' as a likely confounder and report the stratified results,")
        print(f"     not just the marginal one.")
    return out

stratified_analysis(data, measure='risk_difference')

### 3b. Check X–Z and X–Y associations (diagnosing *why* the paradox happens)

As in Section 1.3.2, a Yule–Simpson reversal happens when the candidate
confounder `X` is associated with **both** the treatment `Z` and the outcome `Y`.

In [ ]:
def confounder_diagnostics(df, z_col='Z', y_col='Y', x_col='X'):
    """Report pr(Z=1|X) by level of X, and pr(Y=1|Z,X) by level of X and Z."""
    xz = df.groupby(x_col)[z_col].mean().rename('pr(Z=1 | X)')
    print('X–Z relationship (is X associated with treatment assignment?):')
    print(xz, '\n')

    xy = df.groupby([z_col, x_col])[y_col].mean().rename('pr(Y=1 | Z, X)')
    print('X–Y relationship within each treatment level (is X associated with outcome?):')
    print(xy)
    return xz, xy

_ = confounder_diagnostics(data)

## 4. Regression comparison template (Section 1.2.1)

For a continuous outcome, compare a **naive regression** (outcome on treatment
only) to a **covariate-adjusted regression** (outcome on treatment + controls).
A large change in the treatment coefficient -- or a sign flip -- is the
regression analogue of the Yule–Simpson Paradox.

In [ ]:
# Replace with your own DataFrame containing a continuous outcome and covariates.
# Example structure (uncomment and adapt):

# df = pd.read_csv('your_data.csv')
# outcome = 'y'          # continuous outcome, e.g. 're78'
# treatment = 'treat'    # treatment/exposure indicator
# covariates = ['age', 'education', 'married', 'nodegree']  # candidate confounders
#
# naive = smf.ols(f'{outcome} ~ {treatment}', data=df).fit()
# adjusted = smf.ols(f'{outcome} ~ {treatment} + ' + ' + '.join(covariates), data=df).fit()
#
# comparison = pd.DataFrame({
#     'naive_coef': [naive.params[treatment]],
#     'naive_pvalue': [naive.pvalues[treatment]],
#     'adjusted_coef': [adjusted.params[treatment]],
#     'adjusted_pvalue': [adjusted.pvalues[treatment]],
# })
# comparison['sign_flip'] = np.sign(comparison['naive_coef']) != np.sign(comparison['adjusted_coef'])
# comparison

print('Fill in your own outcome/treatment/covariates above and uncomment to run.')

## 5. Visualize the geometry of Simpson's Paradox (Section 1.3.3, Figure 1.2)

Plots (# failures, # successes) points for each treatment arm, marginally and
within each stratum, so you can see the parallelogram/slope-reversal geometry.

In [ ]:
def plot_simpson_geometry(df, z_col='Z', y_col='Y', x_col='X'):
    fig, ax = plt.subplots(figsize=(6, 6))

    def success_fail_point(sub, z_val):
        s = sub[(sub[z_col] == z_val) & (sub[y_col] == 1)].shape[0]
        f = sub[(sub[z_col] == z_val) & (sub[y_col] == 0)].shape[0]
        return f, s  # (x=failures, y=successes)

    # marginal
    A = success_fail_point(df, 1)
    B = success_fail_point(df, 0)
    ax.plot([0, A[0]], [0, A[1]], 'o-', color='tab:blue', label='Z=1 (marginal)')
    ax.plot([0, B[0]], [0, B[1]], 'o-', color='tab:red', label='Z=0 (marginal)')

    colors = plt.cm.tab10.colors
    for i, (level, sub) in enumerate(df.groupby(x_col)):
        A1 = success_fail_point(sub, 1)
        B1 = success_fail_point(sub, 0)
        ax.plot([0, A1[0]], [0, A1[1]], '^--', color=colors[i % 10], alpha=0.7,
                 label=f'Z=1, {x_col}={level}')
        ax.plot([0, B1[0]], [0, B1[1]], 'v--', color=colors[i % 10], alpha=0.4,
                 label=f'Z=0, {x_col}={level}')

    ax.set_xlabel('# failures')
    ax.set_ylabel('# successes')
    ax.set_title("Geometry of the Yule–Simpson Paradox")
    ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.02, 1))
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.show()

plot_simpson_geometry(data)

## 6. Project checklist (quick reference)

See the companion document *Applicability Checklist* for the full version. In short, before reporting a marginal association as if it were causal or definitive:

1. List candidate confounders you have measured.
2. Run `stratified_analysis()` for each candidate confounder.
3. Run `confounder_diagnostics()` to see if X relates to both Z and Y.
4. If signs flip or magnitudes change a lot, report stratified results and flag the confounding risk.
5. Remember: nothing in this chapter's tools *proves* causation -- they only describe association. Formal causal identification (later chapters) is needed to justify a causal interpretation.